### Setup

In [1]:
suppressPackageStartupMessages({
  library(tidyverse)
  library(readr)
  library(stringr)
  library(cowplot)
  library(scales)
  library(grid)
})

## ---------- paths ----------
data_dir  <- "../data"
out_dir <- "../figs"

path_mut_metagen <- file.path(data_dir, "mutation_metagen_ns.tsv")
path_pca_rnaseq  <- file.path(data_dir, "rna_seq_pca_scores_ancestral_vs_preexposed.csv")
out_pdf <- file.path(out_dir, "Fig6.pdf")

## ---------- load taxonomy from source file ----------
source("project_settings.R")

### Colours and helpers

In [2]:
## ---------- colours ----------
AMP_COL <- "#6D2682"

ab_cols <- c(
  "0"   = "#BBBBBC",
  "5"   = "#AD8CB999",
  "50"  = "#AD8CB9",
  "500" = "#6D2682"
)

## ---------- factor helpers ----------
primer_pulse_levels <- c("0", "5", "50", "500")
factor_primer_pulse <- function(x) factor(as.character(x), levels = primer_pulse_levels)
evo_simple_levels <- c("All ancestral", "All pre-exposed")

## ---------- helpers ----------
is_hypo <- function(x) {
  y <- tolower(trimws(as.character(x)))
  is.na(y) | y == "" |
    str_detect(y, "hypothetical|unknown|uncharacterized|predicted|putative")
}

hambi_to_strainID_dash <- function(h) {
  hh <- as.integer(gsub("\\D", "", as.character(h)))
  sprintf("HAMBI-%04d", hh)
}

abbr_taxon <- function(genus, species) {
  ifelse(
    is.na(genus) | is.na(species) | genus == "" | species == "",
    NA_character_,
    paste0(substr(genus, 1, 1), ". ", species)
  )
}

to_italic_expr <- function(x) {
  x <- as.character(x)
  x <- gsub("\\\\", "\\\\\\\\", x)
  x <- gsub("'", "\\\\'", x)
  parse(text = paste0("italic('", x, "')"))
}

## ---------- shared scales for PCA panels ----------
scale_pre_pulse_colour <- scale_colour_manual(
  values = ab_cols,
  labels = function(x) as.character(x),
  name   = "Ampicillin pre-pulse\npriming (\u00B5g ml\u207B\u00B9)"
)

scale_evo_shape <- scale_shape_manual(
  values = c(
    "Ancestral community" = 16,
    "Full primed" = 17
  ),
  breaks = c("Ancestral community", "Full primed"),
  labels = c("Ancestral community", "Full primed"),
  name   = NULL
)

ellipse_layers <- function(df_all_pre, df_anc_non0, df_anc_0) {
  list(
    stat_ellipse(data = df_all_pre,  aes(x = PC1, y = PC2),
                 type = "norm", level = 0.68, linetype = "solid",
                 size = 0.4, colour = "black", alpha = 0.45, show.legend = FALSE),
    stat_ellipse(data = df_anc_non0, aes(x = PC1, y = PC2),
                 type = "norm", level = 0.68, linetype = "dashed",
                 size = 0.4, colour = "black", alpha = 0.45, show.legend = FALSE),
    stat_ellipse(data = df_anc_0,    aes(x = PC1, y = PC2),
                 type = "norm", level = 0.68, linetype = "dotted",
                 size = 0.4, colour = "black", alpha = 0.45, show.legend = FALSE)
  )
}

### Load data

In [3]:
mut_metagen <- read_tsv(path_mut_metagen, show_col_types = FALSE)

### Panel A

In [4]:
hf <- mut_metagen %>%
  distinct(HAMBI, GENE, gene_name1, Primer_pulse, Evo, Replicate, evo_presence) %>%
  filter(evo_presence == "no") %>%
  count(HAMBI, GENE, gene_name1, Primer_pulse, Evo, name = "COUNT") %>%
  mutate(
    Primer_pulse_f = factor_primer_pulse(Primer_pulse),
    Evo = factor(
      Evo,
      levels = c("All ancestral","Pre-exposed 1972","Pre-exposed 1977","All pre-exposed"),
      labels = c("Anc. comm.", "Dom. species\nprimed", "Subdom. species\nprimed", "Full primed")
    ),
    strainID = hambi_to_strainID_dash(HAMBI)
  ) %>%
  left_join(tax, by = "strainID") %>%
  mutate(
    tax_abbr = abbr_taxon(genus, species),
    gene_label_core = ifelse(is_hypo(gene_name1), as.character(GENE), as.character(gene_name1)),
    row_label_plain = paste0(tax_abbr, " ", gene_label_core)
  )

ord_tbl_A <- hf %>%
  group_by(tax_abbr, row_label_plain) %>%
  summarise(maxCOUNT = max(COUNT), .groups = "drop") %>%
  arrange(tax_abbr, maxCOUNT, row_label_plain)

y_levels_A <- unique(ord_tbl_A$row_label_plain) %>% rev()

hit_vals  <- sort(unique(hf$COUNT))
fill_vals <- alpha(AMP_COL, seq(0.35, 1, length.out = length(hit_vals)))
names(fill_vals) <- as.character(hit_vals)

y_num_tbl <- tibble(row_label_plain = y_levels_A) %>% mutate(y_num = row_number())
hf_A <- hf %>% left_join(y_num_tbl, by = "row_label_plain") %>% filter(!is.na(y_num))

y_max <- max(y_num_tbl$y_num)
pad_y <- 0.25

panel_A <- ggplot(hf_A, aes(x = Primer_pulse_f, y = y_num, fill = factor(COUNT))) +
  geom_tile(colour = "white", linewidth = 0.15) +
  scale_fill_manual(values = fill_vals, name = "Replicates\nhit (non-syn)") +
  scale_x_discrete(
    labels = c("0"="0","5"="5","50"="50","500"="500"),
    drop = FALSE,
    expand = expansion(add = c(0, 0))
  ) +
  scale_y_continuous(
    breaks = y_num_tbl$y_num,
    labels = to_italic_expr(y_levels_A),
    expand = expansion(add = c(0, 0))
  ) +
  facet_grid(. ~ Evo) +  # <-- top strips by default (older ggplot2)
  labs(x = "Ampicillin pre-pulse priming (\u00B5g ml\u207B\u00B9)", y = NULL) +
  coord_fixed(
    ratio = 0.36,
    ylim  = c(1 - pad_y, y_max + pad_y),
    clip  = "on"
  ) +
  theme_minimal(base_size = 9) +
  theme(
    panel.grid          = element_blank(),
    panel.border        = element_rect(colour = "black", fill = NA, linewidth = 0.3),
    axis.text.x         = element_text(angle = 0, vjust = 0.5, hjust = 0.5, size = 8),
    axis.title.x        = element_text(size = 9, face = "bold", margin = margin(t = 6)),
    axis.text.y         = element_text(size = 5.5, margin = margin(r = -10)),
    strip.placement     = "outside",
    strip.background    = element_blank(),
    strip.text.x        = element_text(face = "bold", size = 8, lineheight = 0.95),
    plot.margin         = margin(t = 6, r = 0, b = 0, l = 0),
    panel.spacing.x     = unit(0.05, "lines")
  )

legend_source_A <- panel_A +
  theme(
    legend.position  = "right",
    legend.title     = element_text(size = 8),
    legend.text      = element_text(size = 8),
    legend.margin    = margin(0, 0, 0, 0),
    legend.box.margin = margin(0, 0, 0, 0)
  ) +
  guides(fill = guide_legend(ncol = 1, byrow = TRUE))

legend_A <- cowplot::get_legend(legend_source_A)

## Add a small spacer between heatmap and legend
A_row <- cowplot::plot_grid(
  panel_A + theme(legend.position = "none"),
  NULL,
  legend_A,
  nrow = 1,
  rel_widths = c(1, 0.02, 0.13),  # <-- small gap + slightly larger legend space
  align = "h", axis = "tb"
)

A_labeled <- cowplot::plot_grid(
  A_row,
  labels = "A", label_size = 13, label_fontface = "bold",
  label_x = 0.01, label_y = 1.00, hjust = 0, vjust = 1
)

### Panels B + C 

In [5]:
evo_simple_labels <- c(
  "All ancestral"   = "Ancestral community",
  "All pre-exposed" = "Full primed"
)

mut_for_pca <- mut_metagen %>% filter(Evo %in% evo_simple_levels)
mut_design  <- mut_for_pca %>% distinct(Sample, Evo, Primer_pulse)

mut_matrix <- mut_for_pca %>%
  distinct(Sample, uniqvar) %>%
  mutate(present = 1L) %>%
  pivot_wider(names_from = uniqvar, values_from = present, values_fill = 0) %>%
  arrange(Sample)

mut_mat <- mut_matrix %>% select(-Sample) %>% as.matrix()
mut_pca <- prcomp(mut_mat, center = TRUE, scale. = FALSE)

var_exp     <- mut_pca$sdev^2
var_prop    <- var_exp / sum(var_exp)
pc1_lab_mut <- sprintf("PC1 (%.1f%%)", 100 * var_prop[1])
pc2_lab_mut <- sprintf("PC2 (%.1f%%)", 100 * var_prop[2])

mut_scores <- as_tibble(mut_pca$x[, 1:2], .name_repair = "minimal") %>%
  mutate(Sample = mut_matrix$Sample) %>%
  rename(PC1 = 1, PC2 = 2) %>%
  left_join(mut_design, by = "Sample") %>%
  mutate(
    Evo_simple     = factor(Evo, levels = evo_simple_levels, labels = evo_simple_labels),
    Primer_pulse_f = factor_primer_pulse(Primer_pulse)
  )

mut_all_pre  <- mut_scores %>% filter(Evo_simple == "Full primed")
mut_anc_non0 <- mut_scores %>% filter(Evo_simple == "Ancestral community", Primer_pulse_f != "0")
mut_anc_0    <- mut_scores %>% filter(Evo_simple == "Ancestral community", Primer_pulse_f == "0")

panel_B <- ggplot(mut_scores, aes(PC1, PC2, colour = Primer_pulse_f, shape = Evo_simple)) +
  ellipse_layers(mut_all_pre, mut_anc_non0, mut_anc_0) +
  geom_point(size = 2.2, stroke = 0.4) +
  annotate("text", x = -Inf, y = Inf, label = "Mutation profiles",
           hjust = -0.1, vjust = 1.3, size = 3, fontface = "bold") +
  scale_pre_pulse_colour +
  scale_evo_shape +
  labs(x = pc1_lab_mut, y = pc2_lab_mut) +
  theme_minimal(base_size = 9) +
  theme(
    panel.grid      = element_blank(),
    panel.border    = element_rect(colour = "black", fill = NA, linewidth = 0.3),
    legend.position = "right",
    axis.title      = element_text(size = 9),
    axis.text       = element_text(size = 8),
    plot.margin     = margin(t = 0, r = 3, b = 3, l = 3)
  )

pca_rna_raw <- read_csv(path_pca_rnaseq, show_col_types = FALSE)

pca_rna_df <- pca_rna_raw %>%
  transmute(
    PC1          = as.numeric(PC1),
    PC2          = as.numeric(PC2),
    Evo_raw      = as.character(Community),
    Primer_pulse = as.numeric(Treatment)
  ) %>%
  mutate(
    Evo = case_when(
      str_detect(tolower(Evo_raw), "anc")      ~ "All ancestral",
      str_detect(tolower(Evo_raw), "evo|pre")  ~ "All pre-exposed",
      TRUE                                     ~ NA_character_
    ),
    Evo_simple     = factor(Evo, levels = evo_simple_levels, labels = evo_simple_labels),
    Primer_pulse_f = factor_primer_pulse(Primer_pulse)
  ) %>%
  filter(!is.na(Evo_simple))

rna_all_pre  <- pca_rna_df %>% filter(Evo_simple == "Full primed")
rna_anc_non0 <- pca_rna_df %>% filter(Evo_simple == "Ancestral community", Primer_pulse_f != "0")
rna_anc_0    <- pca_rna_df %>% filter(Evo_simple == "Ancestral community", Primer_pulse_f == "0")

panel_C <- ggplot(pca_rna_df, aes(PC1, PC2, colour = Primer_pulse_f, shape = Evo_simple)) +
  ellipse_layers(rna_all_pre, rna_anc_non0, rna_anc_0) +
  geom_point(size = 2.2, stroke = 0.4) +
  annotate("text", x = -Inf, y = Inf, label = "Gene expression\n   profiles",
           hjust = -0.1, vjust = 1.3, size = 3, fontface = "bold") +
  scale_pre_pulse_colour +
  scale_evo_shape +
  labs(x = "PC1 (90.82%)", y = "PC2 (3.24%)") +
  theme_minimal(base_size = 9) +
  theme(
    panel.grid      = element_blank(),
    panel.border    = element_rect(colour = "black", fill = NA, linewidth = 0.3),
    legend.position = "right",
    axis.title      = element_text(size = 9),
    axis.text       = element_text(size = 8),
    plot.margin     = margin(t = 0, r = 3, b = 4, l = 3)
  )

legend_color_src <- panel_B +
  theme(
    legend.position = "bottom",
    legend.title    = element_text(size = 9),
    legend.text     = element_text(size = 8)
  ) +
  guides(shape = "none")
legend_color <- cowplot::get_legend(legend_color_src)

shape_df <- tibble(
  Evo_simple = factor(
    c("Ancestral community", "Full primed"),
    levels = c("Ancestral community", "Full primed")
  )
)

legend_shape_src <- ggplot(shape_df, aes(x = 1, y = Evo_simple, shape = Evo_simple)) +
  geom_point(size = 2.2, stroke = 0.4, colour = "black") +
  scale_evo_shape +
  theme_void(base_size = 9) +
  theme(
    legend.position = "bottom",
    legend.title    = element_blank(),
    legend.text     = element_text(size = 8)
  )
legend_shape <- cowplot::get_legend(legend_shape_src)

legend_BC_combined <- cowplot::plot_grid(
  legend_color,
  legend_shape,
  nrow = 1,
  rel_widths = c(1.4, 1.0)
)

BC_row_raw <- cowplot::plot_grid(
  panel_B + theme(legend.position = "none"),
  panel_C + theme(legend.position = "none"),
  nrow        = 1,
  rel_widths  = c(1, 1),
  labels      = c("B", "C"),
  label_size  = 13,
  label_fontface = "bold",
  label_y = 1.12
)

BC_spacer <- ggdraw() + theme_void()

BC_with_legend <- cowplot::plot_grid(
  BC_row_raw,
  BC_spacer,
  legend_BC_combined,
  ncol        = 1,
  rel_heights = c(1, 0.03, 0.24),
  align       = "v",
  axis        = "lr"
)

Warning message:
“Using `size` aesthetic for lines was deprecated in ggplot2 3.4.0.
ℹ Please use `linewidth` instead.”


### Assemble + save 

In [6]:
gap_AB <- ggdraw() + theme_void()

fig_w <- 5.6
fig_h <- 5.2

main_plot <- cowplot::plot_grid(
  A_labeled,
  gap_AB,
  BC_with_legend,
  ncol = 1,
  rel_heights = c(1.50, 0.075, 1.00),
  align = "v",
  axis = "lr"
)

pdf_dev <- if (capabilities("cairo")) cairo_pdf else "pdf"

ggsave(
  out_pdf,
  main_plot,
  width  = fig_w,
  height = fig_h,
  units  = "in",
  device = pdf_dev
)

message("Saved Fig6.pdf to: ", normalizePath(out_pdf))

Saved Fig6.pdf to: /Users/cairns/Documents/Academic/Biology/2026/Projects/APEX/analysis/Downstream_analysis_polished/figs/Fig6.pdf

